# Energy storage (battery)

The 50G-cell equivalent-circuit pack: open-circuit voltage and resistance fitted to cell data, sag, ageing and
the power-density factor, and how the pack behaves in the sized aircraft.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
for path in (repo_root / "src", repo_root):     # this checkout's package even if another one is installed
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
warnings.filterwarnings("ignore", category=FutureWarning)

import aerosandbox.numpy as np
import aerosandbox.tools.units as u
import matplotlib.pyplot as plt

check_log = []


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua, yellow = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8})

## 1. The reference pack through the mission and the engine-out hover

In [ ]:
# The current reference: every model at its highest fidelity (about 10 minutes from cold).
from examples.halo_sizing import (HaloAssumptions as RefAssumptions, HaloRequirements as RefRequirements,
                                  soc_emergency_floor, soc_minimum, solve_halo_sizing as solve_reference)
t0 = time.time()
ref = solve_reference()
print(f"reference: {ref.mass_takeoff_kg:,.0f} kg = {ref.mass_takeoff_kg / u.lbm:,.0f} lb, "
      f"empty {ref.mass_empty_kg:,.0f} kg, fuel {ref.mass_fuel_kg:,.0f} kg ({(time.time() - t0) / 60:.1f} min)")
check("reference closes with every margin satisfied", ref.min_margin > -1e-6)
check("reference take-off weight 15,179 lb (+- 20)", abs(ref.mass_takeoff_kg / u.lbm - 15179) < 20)

In [ ]:
trace, eo = ref.battery_trace, ref.engine_out_trace
print(f"pack {ref.energy_battery_kWh:.1f} kWh, {dict(ref.powertrain_masses_kg)['battery']:.0f} kg, "
      f"{ref.design.power_max_discharge_battery_W / 1e3:,.0f} kW")
fig, axes = plt.subplots(2, 2, figsize=(11, 5.8), sharex="col", gridspec_kw=dict(width_ratios=[3, 1.2]))
for column, rows, scale, unit, color in ((0, trace, 60.0, "min", blue), (1, eo, 1.0, "s", orange)):
    ax, ax2 = axes[0, column], axes[1, column]
    t = np.array([[q["time_start_s"], q["time_start_s"] + q["duration_s"]] for q in rows]).ravel() / scale
    ax.plot(t, np.array([[q["soc_start"], q["soc_end"]] for q in rows]).ravel(), color=color, lw=2)
    for q in rows:
        x = np.array([q["time_start_s"], q["time_start_s"] + q["duration_s"]]) / scale
        ax2.plot(x, [q["voltage_ocv_V"]] * 2, color=ink_muted, lw=1, ls="--")
        ax2.plot(x, [q["voltage_bus_V"]] * 2, color=color, lw=2.2)
        ax2.plot(x[1], q["voltage_end_V"], "v", color=color, markersize=6)
    ax.axhline(soc_minimum, color=ink_muted, lw=0.8, ls=":")
    ax.axhline(soc_emergency_floor, color=ink_muted, lw=0.8, ls=":")
    ax.set(ylabel="SOC", ylim=(0, 1))
    ax2.set(xlabel=f"time ({unit})", ylabel="pack voltage (V)")
axes[0, 0].set_title("mission (interval means; dashed: open-circuit voltage)", fontsize=10)
axes[0, 1].set_title("engine-out hover from SOC 0.30", fontsize=10)
fig.tight_layout()
plt.show()
check("the engine-out hover ends at the lowest voltage of any point",
      min(q["voltage_end_V"] for q in eo) <= min(q["voltage_end_V"] for q in trace) + 1e-6)
check("SOC stays above the emergency floor", min(q["soc_end"] for q in list(trace) + list(eo)) >= soc_emergency_floor - 1e-6)

## 2. Equivalent-circuit cell model

*From `tier17_battery/battery_verification.ipynb`.*

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import aerosandbox as asb
import aerosandbox.numpy as np
import aerosandbox.tools.units as u
import matplotlib.pyplot as plt

from aircraft_closure.powertrain import cells
from aircraft_closure.powertrain.components.battery import Battery
from aircraft_closure.powertrain.components.battery_ecm import EquivalentCircuitBattery, inr21700_50g_cell
from functools import partial

from examples.halo_sizing import (HaloAssumptions, assumptions_tier12b, requirements_tier12b, soc_emergency_floor,
                                  soc_minimum)
from examples.halo_sizing import solve_halo_sizing as solve_current

# Pinned to the aircraft this tier was sized on: Tier 12b specific-power machines, no hot-day hover. The current
# reference (Tier 13 machines) gives 785 kg with the same pack; see tests/integration/test_halo_battery_ecm.py.
assumptions_tier17 = replace(assumptions_tier12b, battery_model="ecm")
solve_halo_sizing = partial(solve_current, requirements=requirements_tier12b, assumptions=assumptions_tier12b)

check_log = globals().get("check_log", [])


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua, yellow = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
ramp = ["#0d366b", "#184f95", "#256abf", "#3987e5", "#6da7ec", "#9ec5f4"]   # ordinal (blue 700 -> 200)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8,
                     "axes.labelcolor": ink, "xtick.color": ink_muted, "ytick.color": ink_muted})
cell = inr21700_50g_cell()
ocv_table, dcir_table = cells.load_ocv_table(), cells.load_dcir_table()

### 1. Open-circuit voltage: digitized data and fit

The cell model's OCV is the 30 C curve. That temperature has the complete
0–1 range and the largest markers. The model has no OCV temperature term.
Fig. 7 shows that above 30 % SOC the curves for 10–45 C differ by only a few
tens of mV. Only the -10 C curve departs at low SOC, and the paper's own
-10 C test stops at 15 % SOC.

In [ ]:
soc30, ocv30 = ocv_table.at_temperature(30.0)
grid = np.linspace(0, 1, 401)
tabulated = cells.tabulated_ocv_model(30.0)
residual_V = cell.ocv_model.voltage_V(soc30) - ocv30
print(f"degree-7 polynomial vs 30 C data: rms {1e3 * np.sqrt(np.mean(residual_V**2)):.1f} mV, "
      f"max {1e3 * np.max(np.abs(residual_V)):.1f} mV (digitization about +-5 mV)")
check("OCV fit within 25 mV of every 30 C point", np.max(np.abs(residual_V)) < 0.025)
check("OCV fit is monotonic in SOC", np.all(np.diff(cell.ocv_model.voltage_V(grid)) > 0))
check("tabulated OCV recovers its nodes", np.allclose(tabulated.voltage_V(soc30), ocv30, atol=1e-9))
check("reference point: OCV(50 %, 30 C) = 3.72 V +- 0.02", abs(ocv30[10] - 3.72) < 0.02)
print(f"cell energy 4.9 Ah x mean OCV {cell.ocv_model.mean_voltage_V():.3f} V = "
      f"{cell.capacity_As * cell.ocv_model.mean_voltage_V() / 3600:.2f} Wh "
      f"({cell.capacity_As * cell.ocv_model.mean_voltage_V() / 3600 / cell.mass_kg:.0f} Wh/kg cell)")

fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw=dict(width_ratios=[2, 1]))
for (temperature_C, color, marker) in zip((-10, 10, 20, 45), (ramp[0], ramp[2], ramp[4], orange), "ov^s"):
    s, v = ocv_table.at_temperature(float(temperature_C))
    ax.plot(s, v, marker, color=color, markersize=5, markerfacecolor="none", label=f"{temperature_C} C (digitized)")
ax.plot(soc30, ocv30, "D", color=ink, markersize=5, label="30 C (digitized, fitted)")
ax.plot(grid, cell.ocv_model.voltage_V(grid), color=blue, lw=2, label="model: degree-7 polynomial")
ax.plot(grid, tabulated.voltage_V(grid), color=aqua, lw=1.2, ls="--", label="tabulated (B-spline)")
ax.set(xlabel="SOC", ylabel="cell OCV (V)", title="INR21700-50G open-circuit voltage (Paudel et al. Fig. 7)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax_r.axhline(0, color=ink_muted, lw=0.8)
ax_r.plot(soc30, 1e3 * residual_V, "o-", color=blue, markersize=4)
ax_r.set(xlabel="SOC", ylabel="fit - data (mV)", title="residual, 30 C")
fig.tight_layout()

### 2. Resistance: DCIR data (vector) and the 2-RC fit

- **What DCIR measures:** DCIR = delta V / I at the end of a pulse from
  rest. That includes the OCV drift while the pulse discharges the cell:
  dOCV/dSOC x t / Q. The drift doesn't depend on the current, and it is
  removed before fitting.
- **Model:** the model's pulse response is
  R0 + R1 (1 - e^(-t/8 s)) + R2 (1 - e^(-t/43 s)), with
  R_k(SOC, T) = R_k,ref exp(a x + b x^2) (1 + c e^(-(s-0.2)/w_lo) + d e^((s-0.8)/w_hi))
  and x = T_ref/T - 1.
- **Fit:** an `asb.Opti` least squares over all 308 discharge points
  (`cells.fit_resistance_model`).
- **Cross-check:** the paper's power formula,
  P = 2.5 V (OCV - 2.5 V) / DCIR, reproduces the separately digitized Fig. 8
  power curves. That validates both digitizations.

In [ ]:
discharge = dcir_table.select(True)
target_ohm = cells.resistance_dc_corrected_ohm(discharge, cell.ocv_model, cell.capacity_As)
model_ohm = cell.resistance_model.resistance_dc_ohm(discharge.soc, discharge.temperature_C, discharge.duration_pulse_s)
relative = model_ohm / target_ohm - 1
print(f"2-RC fit vs 308 discharge DCIR points: rms {100 * np.sqrt(np.mean(relative**2)):.1f} %, "
      f"max {100 * np.max(np.abs(relative)):.1f} %")
for temperature_C in (-10, 0, 10, 20, 30, 45):
    mask = discharge.temperature_C == temperature_C
    print(f"  {temperature_C:4d} C: rms {100 * np.sqrt(np.mean(relative[mask]**2)):4.1f} %")
check("resistance fit rms below 5 % over all temperatures and pulses", np.sqrt(np.mean(relative**2)) < 0.05)
refit, _ = cells.fit_resistance_model()
check("the Opti refit reproduces the coded coefficients",
      all(abs(a.resistance_ref_ohm / b.resistance_ref_ohm - 1) < 1e-3
          for a, b in zip(refit.branches, cell.resistance_model.branches)))
mask = discharge.temperature_C >= 20
power_W = cells.pulse_power_W(cell.ocv_model.voltage_V(discharge.soc[mask]), discharge.resistance_dc_ohm[mask])
power_error = power_W / discharge.power_pulse_W[mask] - 1
print(f"P = 2.5 (OCV - 2.5) / DCIR vs digitized Fig. 8 power (20-45 C): rms {100 * np.sqrt(np.mean(power_error**2)):.1f} %")
check("the two digitizations agree (pulse power within 1.5 % rms)", np.sqrt(np.mean(power_error**2)) < 0.015)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
soc_line = np.linspace(0.1, 1.0, 181)
for ax, temperature_C in zip(axes, (0, 25, 45)):
    data_T = 30 if temperature_C == 25 else temperature_C
    for duration_s, color, marker in zip((2, 10, 30, 180), (blue, orange, aqua, yellow), "os^D"):
        m = (discharge.temperature_C == data_T) & (discharge.duration_pulse_s == duration_s)
        ax.plot(discharge.soc[m], 1e3 * target_ohm[m], marker, color=color, markersize=5, markerfacecolor="none")
        ax.plot(soc_line, 1e3 * cell.resistance_model.resistance_dc_ohm(soc_line, float(data_T), float(duration_s)),
                color=color, lw=1.8, label=f"{duration_s} s pulse")
    ax.axvspan(0.2, 0.8, color=grid_color, alpha=0.35, lw=0)
    ax.set(xlabel="SOC", title=f"{data_T} C: data (markers) vs model (lines)")
axes[0].set_ylabel("cell discharge DCIR, drift removed (mOhm)")
axes[0].legend(frameon=False, fontsize=8)
axes[2].annotate("shaded: data range\nbelow 0.2: rise continues\nabove 0.8: held", xy=(0.82, 60), fontsize=8, color=ink_muted)
fig.tight_layout()

### 3. The discharge-curve shape the old model missed

Here is one 50G cell scaled by F = 5 at 25 C, at end of life (capacity 0.8,
resistance 1.5). That is the cell in the Halo option below.

- **Left panel:** terminal voltage against discharge power at five SOCs.
  - Each curve is the low-current root of R I^2 - V* I + P = 0 (solid) up
    to P_max = V*^2 / 4R. Past that point the dotted high-current root is
    the branch the constraint V >= V*/2 excludes.
  - The 2.5 V cutoff line marks the usable power at each SOC.
- **Right panel:** a constant-power discharge against SOC.
  - The new model shows the real knee below 20 % SOC.
  - The old constant-OCV `Battery`, scaled to one cell, is a flat line.

In [ ]:
eol = dict(factor_power_density=5.0, factor_capacity_ageing=0.8, factor_resistance_ageing=1.5)
one = EquivalentCircuitBattery(count_series=1, count_parallel=1.0, **eol)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for soc, color in zip((0.9, 0.6, 0.3, 0.15, 0.1), ramp[:5]):
    steady = one.evaluate(0.0, soc)
    v_star, r = float(steady.voltage_driving_V), float(steady.resistance_effective_ohm)
    current_low = np.linspace(0, v_star / (2 * r), 200)
    current_high = np.linspace(v_star / (2 * r), 0.95 * v_star / r, 100)
    ax.plot(current_low * (v_star - current_low * r) / cell.mass_kg / 1e3, v_star - current_low * r, color=color, lw=2,
            label=f"SOC {soc:.2f}")
    ax.plot(current_high * (v_star - current_high * r) / cell.mass_kg / 1e3, v_star - current_high * r, color=color,
            lw=1, ls=":")
    ax.plot(v_star**2 / (4 * r) / cell.mass_kg / 1e3, v_star / 2, "o", color=color, markersize=6)
ax.axhline(2.5, color=orange, lw=1.2, ls="--")
ax.annotate("cell cutoff 2.5 V", xy=(3.3, 2.55), color=orange, fontsize=8)
ax.set(xlabel="discharge power per cell mass (kW/kg)", ylabel="cell terminal voltage (V)",
       title="steady-state P-V curves (F = 5, end of life, 25 C)")
ax.legend(frameon=False, fontsize=8, title="dots: P_max = V*^2/4R", title_fontsize=8, loc="lower left")

soc_line = np.linspace(0.95, 0.05, 181)
power_cell_W = 5 * 4.9 * 3.6                            # 5C at nominal voltage (88 W)
for F, color in ((5.0, blue), (2.0, aqua)):
    c = EquivalentCircuitBattery(count_series=1, count_parallel=1.0, factor_power_density=F,
                                 factor_capacity_ageing=0.8, factor_resistance_ageing=1.5)
    steady = c.evaluate(0.0, soc_line)
    v_star, r = steady.voltage_driving_V, steady.resistance_effective_ohm
    current = (v_star - np.sqrt(v_star**2 - 4 * r * power_cell_W)) / (2 * r)
    ax2.plot(soc_line, v_star - current * r, color=color, lw=2, label=f"equivalent circuit, F = {F:g}")
ax2.plot(soc_line, cell.ocv_model.voltage_V(soc_line), color=ink_muted, lw=1, ls="--", label="OCV")
legacy = Battery()                                     # 800 V pack with R = 0.05 ohm, scaled to one 3.6 V cell
legacy_cell_V = (legacy.voltage_open_circuit_V - power_cell_W * 222 / legacy.voltage_open_circuit_V
                 * legacy.resistance_ohm) / 222
ax2.plot(soc_line, np.full_like(soc_line, legacy_cell_V), color=orange, lw=1.5, ls="-.",
         label="constant-OCV Battery (Tiers 1-12b)")
ax2.axhline(2.5, color=orange, lw=0.8, ls="--")
ax2.invert_xaxis()
ax2.set(xlabel="SOC (discharging)", ylabel="cell terminal voltage (V)",
        title=f"constant {power_cell_W:.0f} W per cell (5C), end of life; F = 2 ends where P > P_max")
ax2.legend(frameon=False, fontsize=8, loc="lower left")
fig.tight_layout()

low = one.evaluate(0.0, 0.1)
check("P_max = V*^2 / 4R is the peak of V I (numerically)",
      abs(max(i * (float(low.voltage_driving_V) - i * float(low.resistance_effective_ohm))
              for i in np.linspace(0, 2 * float(low.voltage_driving_V / low.resistance_effective_ohm), 20001))
          / float(low.power_max_W) - 1) < 1e-6)
check("terminal voltage falls with SOC at constant power, steeply below 20 %",
      float(one.evaluate(20.0, 0.5).voltage_V - one.evaluate(20.0, 0.2).voltage_V) / 0.3
      < float(one.evaluate(20.0, 0.2).voltage_V - one.evaluate(20.0, 0.1).voltage_V) / 0.1)

### 4. The power-density factor

The factor scales only resistance and current rating. Mass and energy stay
those of the 50G: 263 Wh/kg at the cell, 184 Wh/kg in the pack, and
147 Wh/kg at end of life.

The implied cell-level specific power (beginning of life, 25 C, SOC 0.5) is
measured three ways:

- **Matched load:** V*^2/4R, held steady.
- **US DOE pulse power:** at the 2.5 V cutoff, for 10 s from rest and held
  steady.
- **Current rating:** the continuous rating (2C x F) at nominal voltage.

In [ ]:
rows = []
ocv = float(cell.ocv_model.voltage_V(0.5))
for F in (1, 2, 3, 5, 8, 12):
    r_steady = sum(cell.resistance_model.resistances_ohm(0.5, 25.0)) / F
    r_10s = float(cell.resistance_model.resistance_dc_ohm(0.5, 25.0, 10.0)) / F
    rows.append((F, ocv**2 / (4 * r_steady) / cell.mass_kg, 2.5 * (ocv - 2.5) / r_10s / cell.mass_kg,
                 2.5 * (ocv - 2.5) / r_steady / cell.mass_kg, 9.8 * F * 3.6 / cell.mass_kg, 2 * F))
print(f"{'F':>3} {'matched kW/kg':>14} {'DOE 10 s kW/kg':>15} {'DOE steady kW/kg':>17} {'rating kW/kg':>13} {'C-rate':>7}")
for F, matched, pulse, steady, rating, c_rate in rows:
    print(f"{F:>3} {matched / 1e3:>14.2f} {pulse / 1e3:>15.2f} {steady / 1e3:>17.2f} {rating / 1e3:>13.2f} {c_rate:>7.0f}")
print("pack level: x 0.7 (cell-to-pack mass); end of life: resistance x 1.5")
check("factor 1 reproduces the 50G (resistance and 9.8 A rating)",
      EquivalentCircuitBattery(count_series=1, count_parallel=1.0).get_limits().max_discharge_current_A == 9.8)

## 3. The simple battery model (kept as the fastest option)

*From `tier1_powertrain_components/powertrain_verification.ipynb`.*

In [ ]:
import sys
from pathlib import Path

# Make `examples` importable alongside the installed package.
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import aerosandbox as asb
import aerosandbox.numpy as np
import casadi as cas
import matplotlib.pyplot as plt
from aerosandbox.library.propulsion_propeller import propeller_shaft_power_from_thrust

from aircraft_closure.powertrain.components.motor import Motor, SimpleMotorLossModel
from aircraft_closure.powertrain.components.generator import Generator
from aircraft_closure.powertrain.components.battery import Battery
from aircraft_closure.powertrain.components.turboshaft import SimpleTurboshaft
from aircraft_closure.powertrain.components.gearbox import Gearbox
from aircraft_closure.powertrain.components.propulsor import ActuatorDiskPropulsor
from examples.series_hybrid_point_explicit import solve_reference_point

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})

check_log = globals().get("check_log", [])

def check(name, condition):
    passed = bool(condition)
    check_log.append((name, passed))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

def close(a, b, rel=1e-9, abs_=1e-9):
    return np.all(np.abs(np.asarray(a) - np.asarray(b)) <= abs_ + rel * np.abs(np.asarray(b)))

### 3. Battery

Constant OCV + series resistance. Positive current discharges.
$V = V_{oc} - I R$, $P_{chem} = V_{oc} I = P_e + I^2 R$,
$\text{SOC}_{next} = \text{SOC} - P_{chem}\,\Delta t / E$. No SOC clamping inside the model.
Mass is the larger of the energy-sized and power-sized pack.

In [ ]:
battery = Battery()
b = battery.evaluate(100, 0.9, 60)
print(b)

check("battery: terminal voltage = V_oc - I R", b.voltage_V == 800 - 100 * 0.05)
check("battery: P_chem = P_elec + P_loss", close(b.power_chemical_W, b.power_electric_W + b.power_loss_W))
check("battery: SOC drop matches chemical energy",
      close((0.9 - b.soc_next) * battery.energy_capacity_J, b.power_chemical_W * 60))
c = battery.evaluate(-100, 0.5, 60)
check("battery: charging -> negative terminal power, SOC rises, loss still positive",
      c.power_electric_W < 0 and c.soc_next > 0.5 and c.power_loss_W > 0)
z = battery.evaluate(0, 0.5, 60)
check("battery: zero current -> OCV, no loss, SOC unchanged",
      z.voltage_V == 800 and z.power_loss_W == 0 and z.soc_next == 0.5)
check("battery: no hidden SOC clamping", battery.evaluate(100, 0.2, 1000).soc_next < battery.min_soc)
ideal = Battery(resistance_ohm=0).evaluate(10000 / 800, 0.9, 360)
check("battery: ideal 10 kW for 360 s uses 0.1 SOC of 36 MJ", close(ideal.soc_next, 0.8))

In [ ]:
current_A = np.linspace(-1000, 8000, 400)
sweep = battery.evaluate(current_A, 0.9)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(current_A, sweep.voltage_V)
ax1.set(xlabel="current [A] (+ discharge)", ylabel="terminal voltage [V]", title="Polarization line")
ax2.plot(current_A, sweep.power_electric_W / 1e3, label="terminal")
ax2.plot(current_A, sweep.power_chemical_W / 1e3, "--", label="chemical")
ax2.axhline(battery.max_discharge_power_W / 1e3, color="k", lw=0.8, label="max discharge rating")
ax2.set(xlabel="current [A]", ylabel="power [kW]", title="Power vs current")
ax2.legend()
plt.show()

# Terminal power peaks at I = V_oc / (2R) (matched load), giving V_oc^2 / (4R).
current_peak_A = current_A[np.argmax(sweep.power_electric_W)]
check("battery: terminal power peaks near V_oc/(2R)",
      abs(current_peak_A - 800 / (2 * 0.05)) < (current_A[1] - current_A[0]))

In [ ]:
# Mass sizing: energy-sized vs power-sized pack.
power_W = np.linspace(0, 400000, 200)
mass_kg = Battery(max_discharge_power_W=power_W).get_mass()
fig, ax = plt.subplots()
ax.plot(power_W / 1e3, mass_kg, label="get_mass()")
ax.axhline(battery.energy_capacity_J / battery.specific_energy_J_kg, ls="--", color="gray", label="energy-sized")
ax.plot(power_W / 1e3, power_W / battery.specific_power_W_kg, ls=":", color="gray", label="power-sized")
ax.set(xlabel="max discharge power [kW]", ylabel="pack mass [kg]", title="Battery mass: max(energy, power)")
ax.legend()
plt.show()

check("battery: default mass is energy-sized (40 kg)", Battery().get_mass() == 40)
check("battery: 300 kW discharge rating -> power-sized 100 kg", Battery(max_discharge_power_W=300000).get_mass() == 100)
check("battery: 600 kW charge rating -> power-sized 200 kg", Battery(max_charge_power_W=600000).get_mass() == 200)

## Summary

In [ ]:
failed = [name for name, passed in check_log if not passed]
print(f"{len(check_log) - len(failed)} of {len(check_log)} checks passed")
for name in failed:
    print("FAILED:", name)